# Parallel Calculations over Compositions

A single `equilibrium()` call runs on one CPU core. To use several cores for a large composition grid, split the grid into chunks and solve each chunk in a separate **process**, then join the results back into one `Dataset`.

Threads (`concurrent.futures.ThreadPoolExecutor`) do not help here: the solver holds Python's global interpreter lock (GIL) while it runs, so threads take turns rather than running at the same time. In our tests, a thread pool was slower than solving the chunks one after another.

In [1]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
from importlib.resources import files

import numpy as np
import xarray as xr

from pycalphad import Database, equilibrium, variables as v
import pycalphad.tests.databases

dbf = Database(files(pycalphad.tests.databases) / "alcrni.tdb")
comps = ["AL", "CR", "NI", "VA"]
phases = list(dbf.phases.keys())

## Split the composition grid into chunks

Here the grid is 40 × 40 compositions in Al-Cr-Ni at 1273 K. We split the `X(CR)` axis into chunks; each chunk is an ordinary conditions dictionary that could be passed to `equilibrium()` on its own. A few chunks per worker keeps all the workers busy even if some chunks take longer than others.

In [2]:
x_cr = np.linspace(0.01, 0.59, 40)
x_ni = np.linspace(0.01, 0.39, 40)
base_conditions = {v.T: 1273, v.P: 101325, v.N: 1, v.X("NI"): x_ni}

n_workers = 2
n_chunks = 4 * n_workers
chunk_conditions = [{**base_conditions, v.X("CR"): chunk} for chunk in np.array_split(x_cr, n_chunks)]
[len(conds[v.X("CR")]) for conds in chunk_conditions]

[5, 5, 5, 5, 5, 5, 5, 5]

## Solve the chunks in a process pool

`functools.partial` fixes the database, components and phases, so each task only needs its conditions. Because `equilibrium` is imported from `pycalphad` (not defined in this notebook), it can be sent to the worker processes.

We ask for the `"spawn"` start method explicitly so the example behaves the same on Linux, macOS and Windows. Each spawned worker imports pycalphad before it can start, which takes a few seconds. `as_completed` hands back chunks as they finish, which is where a progress bar (for example `tqdm`, if you have it installed) or a simple message can go.

In [3]:
solve_chunk = partial(equilibrium, dbf, comps, phases)
results = [None] * n_chunks

context = multiprocessing.get_context("spawn")
with ProcessPoolExecutor(max_workers=n_workers, mp_context=context) as executor:
    futures = {executor.submit(solve_chunk, conds): i for i, conds in enumerate(chunk_conditions)}
    for n_done, future in enumerate(as_completed(futures), start=1):
        results[futures[future]] = future.result()
        print(f"{n_done}/{n_chunks} chunks done")

1/8 chunks done
2/8 chunks done
3/8 chunks done
4/8 chunks done


5/8 chunks done
6/8 chunks done
7/8 chunks done
8/8 chunks done


## Combine the results

Every chunk returns an `xarray.Dataset` with the same dimensions, so they can be joined along the chunked condition, `X_CR`. As a check, the combined result matches a single `equilibrium()` call over the whole grid.

In [4]:
eq = xr.concat(results, dim="X_CR")

eq_single = equilibrium(dbf, comps, phases, {**base_conditions, v.X("CR"): x_cr})
np.testing.assert_allclose(eq.GM.values, eq_single.GM.values)
eq

<xarray.Dataset> Size: 846kB
Dimensions:    (N: 1, P: 1, T: 1, X_CR: 40, X_NI: 40, vertex: 4, component: 3,
                internal_dof: 8)
Coordinates:
  * N          (N) float64 8B 1.0
  * P          (P) float64 8B 1.013e+05
  * T          (T) float64 8B 1.273e+03
  * X_CR       (X_CR) float64 320B 0.01 0.02487 0.03974 ... 0.5603 0.5751 0.59
  * X_NI       (X_NI) float64 320B 0.01 0.01974 0.02949 ... 0.3705 0.3803 0.39
  * vertex     (vertex) int64 32B 0 1 2 3
  * component  (component) <U2 24B 'AL' 'CR' 'NI'
Dimensions without coordinates: internal_dof
Data variables:
    NP         (N, P, T, X_CR, X_NI, vertex) float64 51kB 1.0 nan ... nan nan
    GM         (N, P, T, X_CR, X_NI) float64 13kB -6.639e+04 ... -6.485e+04
    MU         (N, P, T, X_CR, X_NI, component) float64 38kB -6.41e+04 ... -7...
    X          (N, P, T, X_CR, X_NI, vertex, component) float64 154kB 0.98 .....
    Y          (N, P, T, X_CR, X_NI, vertex, internal_dof) float64 410kB 0.98...
    Phase      (N, P, T, X_CR, X_NI, vertex) <U7 179kB 'LIQUID' '' '' ... '' ''
Attributes:
    engine:   pycalphad 0.11.3.dev24+g6f9e2ce60.d20261017
    created:  2026-10-17T01:11:30.836770

## When is this worth it?

The worker start-up cost is paid once per worker, so processes only pay off when the serial calculation takes much longer than that start-up. The small grid above is chosen to run quickly and is not large enough to be faster in parallel. Rough figures from an Apple M4 laptop (4 performance + 6 efficiency cores), with 4 workers and 16 chunks:

| Grid | Serial `equilibrium()` | Process pool (4 workers) | Thread pool (4 threads) |
|---|---|---|---|
| Al-Zn binary, 10,000 points | ~4 s | ~4 s | ~4.5 s |
| Al-Zn binary, 40,000 points | ~14 s | ~6 s | ~17 s |
| Al-Cr-Ni ternary, 25,600 points | ~20 s | ~9 s | ~25 s |

A single large `equilibrium()` call over the whole grid took about as long as the same grid solved chunk by chunk in serial, so splitting the grid adds little overhead. When you run a process pool from a Python script instead of a notebook, put the pool code under an `if __name__ == "__main__":` guard, which the `"spawn"` start method requires.